<a href="https://colab.research.google.com/github/antonijamatek/antonijamatek/blob/main/eos80_inverse_model_UUPSS_Pag_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import needed packages

In [ ]:
# ============================================================
# Setup Cell — Run First
# Automatically prepares the Colab environment
# ============================================================

# Install required packages quietly
!pip install -q pandas numpy

# Import libraries
import math
import pandas as pd
import numpy as np

print("✓ Environment ready")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

#**Section 1: Specific gravity calculation from *in situ* measured Baume degrees [°Be]**

In this section we convert Baume degrees [°Be] to specific gravity [sg].



**1. Convert Baume readings**

Convert observed Baumé readings to specific gravity using the standardized heavy-liquid Baumé relation referenced to 60 °F/60 °F

Reference: *U.S. Bureau of Standards (1916). Circular No. 19: Standard Density and Volumetric Tables(5th ed.). Washington, DC: Government Printing Office*

In [ ]:
def baume_to_sg_observed(baume_obs):
    """
    Convert observed heavy-liquid Baumé reading to observed specific gravity.

    Formula:
    sg_obs = 145 / (145 - Be_obs)
    """
    return 145 / (145 - baume_obs)

In [ ]:
# Observed Baume value
baume_obs = 25.4

#Implement function set in the previuos cells
sg_obs = baume_to_sg_observed(baume_obs)

#Show results
print("Observed Baumé:", baume_obs)
print("Observed specific gravity:", sg_obs)

**2. Correct to the hydrometer calibration temperature of 15.6 °C**

Reference: *temperature-correction guidance based on CRC Handbook of Physics and Chemistry and on Bonython's research for ICI (1948) - https://en.wikibooks.org/wiki/Methods_Manual_for_Salt_Lake_Studies/Salinity/measuring_brine_density*

In [ ]:
#SET THE RIGHT K VALUE

# Observed sample temperature (°C)
temp_obs = 24

# Hydrometer calibration temperature (°C)
temp_ref = 15.65

# Determine K based on SG range
if sg_obs < 1.100:
    K = 5

elif sg_obs < 1.200:
    K = 3

else:
    K = 2

print("K value:", K)

In [ ]:
# FUNCTION TO CORRECT OBSERVED SPECIFIC GRAVITY TO 15.6 °C

def sg_corrected_to_15_6(sg_obs, temp_obs, K):

    return sg_obs + 0.001 * (temp_obs - 15.65) / K

Implement correction and get final specific gravity.

In [ ]:
sg_fin = sg_corrected_to_15_6(sg_obs, 22, K)

print("Specific gravity corrected to 15.6 °C:", sg_fin)

#**2. Section: EOS 80 Density Function**

The function density_eos80(S, T, p_dbar) is a forward model. It takes salinity S, temperature T, and pressure p, and returns the density predicted by the EOS-80 seawater equation of state.

To estimate salinity from a measured density, you then wrap this function inside a root-finding algorithm such as bisection.

**1. FORWARD MODEL**

The code implements a legacy seawater equation of state called EOS-80. It first computes density at atmospheric pressure, then corrects that density for compression under pressure using the secant bulk modulus.

This is the same overall structure used in UNESCO/Fofonoff-Millard style seawater calculations.



In [ ]:
def density_eos80(S, T, p_dbar):
    """
    Compute in-situ density [kg/m^3] from:
        S      : salinity [approx. PSU / g/kg-like practical input]
        T      : temperature [deg C]
        p_dbar : pressure [dbar]

    Uses EOS-80 / UNESCO-style polynomial density and secant bulk modulus.

    NOTE:
    - This formulation is standard for seawater, not extreme brines.
    - For very high salinities (e.g. > 60-70), results are only approximate.
    """

    # Convert pressure from dbar to bar for EOS-80 formulas
    p_bar = p_dbar / 10.0

    # Pure water density at atmospheric pressure
    rho_w = (
        999.842594
        + 6.793952e-2 * T
        - 9.095290e-3 * T**2
        + 1.001685e-4 * T**3
        - 1.120083e-6 * T**4
        + 6.536332e-9 * T**5
    )

    # Salinity correction coefficients A, B, C
    A = (
        0.824493
        - 4.0899e-3 * T
        + 7.6438e-5 * T**2
        - 8.2467e-7 * T**3
        + 5.3875e-9 * T**4
    )

    B = (
        -5.72466e-3
        + 1.0227e-4 * T
        - 1.6546e-6 * T**2
    )

    C = 4.8314e-4

    S_sqrt = math.sqrt(S) if S >= 0 else float("nan")

    # Density at atmospheric pressure calculated using coefficients A, B, C
    rho0 = rho_w + A * S + B * S * S_sqrt + C * S**2

    # Secant bulk modulus K(S, T, p) - coefficients K_w, F, G
    K_w = (
        19652.21
        + 148.4206 * T
        - 2.327105 * T**2
        + 1.360477e-2 * T**3
        - 5.155288e-5 * T**4
    )

    F = (
        54.6746
        - 0.603459 * T
        + 1.09987e-2 * T**2
        - 6.1670e-5 * T**3
    )

    G = (
        7.944e-2
        + 1.6483e-2 * T
        - 5.3009e-4 * T**2
    )

  # The secant bulk modulus at zero pressure K0
    K0 = K_w + F * S + G * S * S_sqrt

  # Pressure dependent coefficients A_p and B_w
    A_w = (
        3.239908
        + 1.43713e-3 * T
        + 1.16092e-4 * T**2
        - 5.77905e-7 * T**3
    )

    A_p = (
        A_w
        + (2.2838e-3 - 1.0981e-5 * T - 1.6078e-6 * T**2) * S
        + 1.91075e-4 * S * S_sqrt
    )

    B_w = (
        8.50935e-5
        - 6.12293e-6 * T
        + 5.2787e-8 * T**2
    )

    B_p = B_w + (-9.9348e-7 + 2.0816e-8 * T + 9.1697e-10 * T**2) * S

    # Final secant bulk modulus at specified salinity, temperature and pressure
    K = K0 + A_p * p_bar + B_p * p_bar**2

    # In-situ density - applies the pressure correction. This is the standard EOS-80 way to move from atmospheric-pressure density to in-situ density.
    rho = rho0 / (1.0 - p_bar / K)
    return rho


**2. INVERSE MODEL**

Computes salinity we need using inverse bisection method (using numerical root-finding algorithm).

Mathematically, the eos80 function computes a forward map:

*density_eos80(S, T, p) -> rho*

Once this forward map is available, salinity can be recovered from measured density by solving the inverse problem:

*density_eos80(S, T, p) - rho_obs = 0*


Salinity is recovered by solving *f(S) = density_eos80(S,T,p) - rho_obs* on a chosen interval <s_min, s_max> (parameter space was enlarged because of brines - from 0 to 300)

In [ ]:
def salinity_from_density(rho_obs, T, p_dbar, s_min=0.0, s_max=500.0, tol=1e-6, max_iter=200):
    """
    Estimate salinity from observed density [kg/m^3], temperature [deg C],
    and pressure [dbar] using bisection.

    Parameters
    ----------
    rho_obs : float
        Observed density [kg/m^3]
    T : float
        Temperature [deg C]
    p_dbar : float
        Pressure [dbar]
    s_min : float
        Lower salinity bound, default 0
    s_max : float
        Upper salinity bound, default 300
    tol : float
        Convergence tolerance in salinity
    max_iter : int
        Maximum iterations

    Returns
    -------
    S_est : float
        Estimated salinity
    """

    # INVERSE MODEL : Define a residual function: f(S) = density_eos80(S, T, p) - rho_obs
    #If f(S) = 0, then the predicted density matches the observed density, so that S is the estimated salinity.


    def f(S):
        return density_eos80(S, T, p_dbar) - rho_obs

    #THE BISECTION METHOD
    #The bisection method starts from an interval [s_min, s_max], for example [0, 300].
    #It requires the residual function to have opposite signs at the two endpoints.
    #At each iteration it evaluates the midpoint, keeps the half-interval that still contains the root, and repeats.
    #The interval becomes smaller and smaller until the salinity estimate is accurate enough.


    f_min = f(s_min)
    f_max = f(s_max)

    if math.isnan(f_min) or math.isnan(f_max):
        raise ValueError("Density function returned NaN at the search bounds.")

    if f_min == 0:
        return s_min
    if f_max == 0:
        return s_max

    # Need a sign change for bisection
    if f_min * f_max > 0:
        raise ValueError(
            f"No root found in [{s_min}, {s_max}]. "
            f"Try checking rho_obs or expanding bounds."
        )

    lo, hi = s_min, s_max

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        f_mid = f(mid)

        if abs(hi - lo) < tol or abs(f_mid) < 1e-10:
            return mid

        if f_min * f_mid < 0:
            hi = mid
            f_max = f_mid
        else:
            lo = mid
            f_min = f_mid

    return 0.5 * (lo + hi)

**3. IMPLEMENT INVERSE MODEL**

Implement inverse model to estimate salinity based on measured specific gravity corrected to the hydrometer calibration temperature of 15.6 °C (rho_obs) and measured temperature.


In [ ]:
rho_obs = sg_fin * 1000         # density [kg m-3] computed from calculated specific gravity in section 1
T = temp_ref        # deg C
p = 0.0            # dbar

S = salinity_from_density(rho_obs, T, p, s_min=0, s_max=500)
print(f"Estimated salinity: {S:.6f}")